Tienes la variable objetivo (ratio), pero para que el modelo aprenda necesita predictores. Ahora mismo tienes tipo de fase, material, familia, cantidad, dedicación... Hay que repasar qué features tienes y cuáles derivar (por ejemplo, el rendimiento histórico de cada familia de fase en cada máquina, que suele ser lo más predictivo). Ese ejercicio decide si tu proyecto es rico, y responde tu vieja preocupación de "si da para IA".

Tu problema es de idoneidad fase-máquina. El modelo predice el ratio a partir de características disponibles antes de ejecutar la fase. Esa última parte es la regla de oro — todo lo que uses como predictor tiene que ser información que existiría en el momento de decidir a qué máquina asignar la fase. Si usas algo que solo se conoce después de ejecutarla, es leakage y el modelo hace trampa.

Leakage es cuando tu modelo aprende usando información que no estaría disponible en el momento real de hacer la predicción. El modelo parece funcionar de maravilla en tus pruebas, pero es un espejismo: está "haciendo trampa" usando datos que en la vida real no tendría. Cuando lo pusieras a funcionar de verdad, fracasaría.

La pregunta que define todo: en el momento en que el planificador decide a qué máquina mandar una fase, ¿qué sabe y qué no sabe? Todo lo que sabe → puede ser predictor. Todo lo que solo se conoce después de ejecutar la fase → es leakage, prohibido.

El caso más obvio en tu proyecto

Tu variable objetivo es el ratio = tiempo_real / tiempo_teórico. El tiempo_real se conoce después de ejecutar la fase. Así que:

Usar tiempo_real_efectivo como predictor sería leakage total y absurdo — estarías usando parte de la respuesta para predecir la respuesta. Es como predecir si un alumno aprobará usando su nota final. El modelo tendría 100% de acierto y sería completamente inútil.

Por la misma razón, dedicacion_media, n_imputaciones, todos los MinutosReal_* → leakage. Todos se conocen solo tras ejecutar. En el momento de asignar, no existen aún.

La features saldrán de lo que ERP/MES ya registran, no de instrumentar la planta

### Predicción de la desviación temporal

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow as pa

In [2]:
df_features = pd.read_parquet("res//procesed/df_FASE_final2.parquet")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)   
display(df_features.head(1))

,SmFaseUid,MaquinaDominante,NombreMaquina,CodigoGrupo,NombreGrupo,smOrdenId,smOrdenNme,smFaseNme,IDUsualTask,smMaterialId,Quantity,tiempo_real_efectivo,tiempo_teorico_total,ratio,ProductEntry,MaterialExit,n_imputaciones
0,00013c31-3895-4370-bb3e-4255371e8d8f,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,TPU,Torns Puma (Taller COURES),OF2509545,Portacápsulas F704501049704(150713),TPP,29783995-2f56-48be-9664-461e6f0b402a,PE1JNCX0012,3.0,41.76667,21.0,1.988889,0.0,0.0,2


In [ ]:
df_features.drop("SmFaseUid", "tiempo_real_efectivo", "tiempo_teorico_total", "ratio", "n_imputaciones")